# Day 5 — Custom Metrics: GEval, BaseMetric, and LatencyMetric

**Module 4 · DeepEval LLM Testing**

---

## What we'll cover

| # | Topic | Why it matters |
|---|---|---|
| 1 | GEval — criteria-driven evaluation | Write quality rubrics in plain English; no Python score logic |
| 2 | BaseMetric — build your own judge | Rule-based checks: keywords, format, length — no LLM needed |
| 3 | LatencyMetric | Slow responses are failing responses; gate on response time |
| 4 | Combining metrics | Production quality gates mix LLM-judge + rule-based checks |

**Prerequisite:** `.env` with `OPENAI_API_KEY` (or Ollama running) and `deepeval` installed (`pip install deepeval`).

---

## The Big Idea

> **GEval is like giving the judge a marking rubric written in plain English.**
> Instead of defining score logic in Python, you write:
> *"A good answer should be concise and on-topic."*
> The judge interprets your rubric and scores accordingly.

Before today's module, you used pre-built metrics like `AnswerRelevancyMetric` — someone else wrote the rubric for you.

Today you learn to **write your own rubric** (GEval), **skip the LLM entirely for deterministic checks** (BaseMetric), and **measure response time as a quality dimension** (LatencyMetric).

---

In [ ]:
# Setup — imports, environment, LLM client
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

from deepeval.metrics import GEval, BaseMetric, LatencyMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

load_dotenv()

# Which provider are we targeting? Set PROVIDER in .env: azure | openai | ollama
PROVIDER = os.getenv("PROVIDER", "ollama").lower()
MODEL    = os.getenv("DEMO_MODEL", "llama3.2:3b")

if PROVIDER == "azure":
    llm_client = OpenAI(
        base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_KEY"),
    )
    MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT", MODEL)
elif PROVIDER == "openai":
    llm_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
else:  # ollama
    llm_client = OpenAI(
        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        api_key="ollama",
    )
    MODEL = os.getenv("OLLAMA_MODEL", MODEL)

def ask(prompt: str, system: str = "You are a helpful assistant.") -> tuple[str, float]:
    """Call the LLM. Returns (response_text, latency_in_seconds)."""
    start = time.perf_counter()
    resp  = llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.3,
        max_tokens=300,
    )
    latency = time.perf_counter() - start
    return resp.choices[0].message.content.strip(), latency

print(f"Provider : {PROVIDER}")
print(f"Model    : {MODEL}")
print("Setup complete.")

---
## 1. GEval — Criteria-Driven Evaluation

### How GEval works

`GEval` sends your rubric + the test case fields to an LLM judge, which returns a score (0–1) and a reason.

**Key parameters:**

| Parameter | What it does |
|---|---|
| `name` | A short label for the metric (e.g. `"Conciseness"`) |
| `criteria` | Plain-English rubric — what makes a response good or bad on this dimension |
| `evaluation_params` | Which fields the judge sees: `INPUT`, `ACTUAL_OUTPUT`, `EXPECTED_OUTPUT`, `CONTEXT` |
| `threshold` | Minimum score (0–1) for `is_successful()` to return `True` |

### LLMTestCaseParams — what you can expose to the judge

```python
LLMTestCaseParams.INPUT           # the user's question
LLMTestCaseParams.ACTUAL_OUTPUT   # the model's response (most common)
LLMTestCaseParams.EXPECTED_OUTPUT # the gold-standard answer (for comparison)
LLMTestCaseParams.CONTEXT         # retrieved chunks / RAG context
```

Most criteria only need `ACTUAL_OUTPUT`. Add `INPUT` when the rubric cares whether the response stays on-topic.

---

In [ ]:
# GEval Metric 1: Conciseness
conciseness_metric = GEval(
    name="Conciseness",
    criteria=(
        "The response should be concise and not ramble or repeat itself. "
        "A high-scoring response gets to the point quickly and avoids unnecessary filler. "
        "A low-scoring response is verbose, repetitive, or padded with re-statements."
    ),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.6,
)

print("Conciseness GEval metric created.")
print(f"  Threshold : {conciseness_metric.threshold}")

In [ ]:
# Test Conciseness on two hand-crafted responses
# (We hard-code them here so you can see the rubric working in isolation,
#  without needing a live LLM to generate them.)

concise_response = (
    "Python is a high-level, interpreted programming language known for its "
    "readable syntax and large standard library."
)

rambling_response = (
    "Well, Python is a programming language. It's a high-level language. "
    "It's also interpreted, which means it's interpreted. "
    "Many people use Python. Python is used by many people around the world. "
    "Python is very popular and a lot of developers use it every day because it is popular. "
    "So yeah, Python is a language that is high-level and interpreted and used by many."
)

cases = [
    ("concise",   concise_response),
    ("rambling",  rambling_response),
]

for label, output in cases:
    tc = LLMTestCase(
        input="What is Python?",
        actual_output=output,
    )
    conciseness_metric.measure(tc)
    status = "PASS" if conciseness_metric.is_successful() else "FAIL"
    print(f"[{label.upper()}] score={conciseness_metric.score:.2f} | {status}")
    print(f"  Reason : {conciseness_metric.reason}")
    print()

In [ ]:
# GEval Metric 2: Formal Tone
formal_tone_metric = GEval(
    name="Formal Tone",
    criteria=(
        "The response must use formal professional English. "
        "No slang, contractions, or casual phrases. "
        "A high-scoring response sounds like a well-written business document. "
        "A low-scoring response contains casual language like 'gonna', 'kinda', "
        "'you know', contractions such as 'don\'t' or 'it\'s', or informal expressions."
    ),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7,
)

formal_response = (
    "The quarterly results indicate a significant improvement in operational efficiency. "
    "Revenue increased by 12 percent compared to the prior period, "
    "driven primarily by growth in the enterprise segment."
)

informal_response = (
    "So basically the numbers are looking pretty good this quarter! "
    "We're up like 12% from last time, which is kinda awesome. "
    "The enterprise guys are really killing it, you know?"
)

tone_cases = [
    ("formal",   formal_response),
    ("informal", informal_response),
]

for label, output in tone_cases:
    tc = LLMTestCase(
        input="Summarise the quarterly results.",
        actual_output=output,
    )
    formal_tone_metric.measure(tc)
    status = "PASS" if formal_tone_metric.is_successful() else "FAIL"
    print(f"[{label.upper()}] score={formal_tone_metric.score:.2f} | {status}")
    print(f"  Reason : {formal_tone_metric.reason}")
    print()

### Writing Good GEval Criteria — Tips

A well-written criterion makes the judge reliable and consistent. Follow these rules:

1. **One dimension per metric.** Don't mix conciseness + tone in one criterion. Create two metrics.

2. **Be specific, not vague.** 
   - Bad: `"The response should be good."`
   - Good: `"The response should answer the question directly without introducing unrelated topics."`

3. **Tell the judge what HIGH and LOW look like.** The judge is scoring on a spectrum. Anchor both ends:
   - `"A score of 1.0 means … A score of 0 means …"`
   - Or: `"High-scoring responses … Low-scoring responses …"`

4. **Avoid negation-only criteria.** Don't just say what the response must NOT do. Also say what it SHOULD do.

5. **Test your rubric on obvious cases first.** A clearly concise response should score ≥ 0.8; a clearly rambling one should score ≤ 0.3. If not, rewrite the criterion.

---

---
## 2. BaseMetric — Write Your Own Judge

### When to use BaseMetric vs GEval

| Situation | Use |
|---|---|
| Checking semantic quality (tone, relevance, completeness) | **GEval** — needs the LLM's judgment |
| Checking whether specific keywords appear | **BaseMetric** — deterministic, zero cost |
| Checking response format (JSON, bullet list, length) | **BaseMetric** — rule-based logic |
| Checking response latency | **LatencyMetric** (built-in BaseMetric subclass) |

**Rule of thumb:** If you could write the check as a simple Python `if` statement, use `BaseMetric`. If you need human-level judgment to assess quality, use `GEval`.

### BaseMetric interface

Subclass `BaseMetric` and implement:
- `measure(test_case)` → sets `self.score` and `self.reason`, returns `self.score`
- `is_successful()` → returns `True` if `self.score >= self.threshold`
- `async a_measure(test_case)` → async wrapper (calls synchronous `measure`)
- `__name__` property → the metric's display name

---

In [ ]:
class KeywordCoverageMetric(BaseMetric):
    """
    Checks what fraction of required_keywords appear in the actual_output.

    score = (number of keywords found) / (total keywords required)

    Use this to enforce that the model mentions specific domain terms,
    required disclosures, or key concepts in its response.
    """

    def __init__(self, required_keywords: list[str], threshold: float = 1.0):
        """
        Args:
            required_keywords: List of keywords that must appear in the response.
                               Matching is case-insensitive.
            threshold: Minimum fraction (0–1) of keywords that must be found.
                       Default 1.0 means ALL keywords must be present.
        """
        self.required_keywords = [kw.lower() for kw in required_keywords]
        self.threshold         = threshold
        self.score: float      = 0.0
        self.reason: str       = ""

    @property
    def __name__(self) -> str:  # type: ignore[override]
        return "KeywordCoverage"

    def measure(self, test_case: LLMTestCase) -> float:
        output_lower = test_case.actual_output.lower()

        found   = [kw for kw in self.required_keywords if kw in output_lower]
        missing = [kw for kw in self.required_keywords if kw not in output_lower]

        total       = len(self.required_keywords)
        self.score  = len(found) / total if total > 0 else 1.0

        found_str   = ", ".join(found)   if found   else "none"
        missing_str = ", ".join(missing) if missing else "none"
        self.reason = (
            f"Found {len(found)}/{total} required keywords. "
            f"Present: [{found_str}]. "
            f"Missing: [{missing_str}]."
        )
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:  # type: ignore[override]
        """Async variant — just delegates to the synchronous measure."""
        return self.measure(test_case)


print("KeywordCoverageMetric defined.")

In [ ]:
# Test KeywordCoverageMetric on three cases
required = ["encryption", "authentication", "firewall", "audit log"]
keyword_metric = KeywordCoverageMetric(required_keywords=required, threshold=0.75)

test_responses = [
    (
        "all found",
        "Our platform uses encryption at rest and in transit. "
        "Multi-factor authentication is enforced for all users. "
        "A firewall filters inbound traffic, and every action is recorded in the audit log."
    ),
    (
        "half found",
        "We use encryption and require authentication for all logins. "
        "Our team is reviewing network policies."
    ),
    (
        "none found",
        "Our product is easy to use and has a great user interface. "
        "Customers love the dashboard."
    ),
]

for label, output in test_responses:
    tc = LLMTestCase(
        input="Describe your security features.",
        actual_output=output,
    )
    keyword_metric.measure(tc)
    status = "PASS" if keyword_metric.is_successful() else "FAIL"
    print(f"[{label.upper()}] score={keyword_metric.score:.2f} | {status}")
    print(f"  {keyword_metric.reason}")
    print()

---
## 3. LatencyMetric

Latency is a quality metric too. If your model takes 30 seconds to respond, users won't use it — no matter how accurate the answer is. `LatencyMetric` gates on response time and fails any test case where `test_case.latency` (in seconds) exceeds your budget.

**How it works:**
1. You call the LLM and record the wall-clock time.
2. You set `test_case.latency = elapsed_seconds` on the `LLMTestCase`.
3. `LatencyMetric(max_seconds=N)` checks `latency <= max_seconds`.

Think of it as an SLA (Service Level Agreement) enforced in your test suite.

---

In [ ]:
# Demonstrate LatencyMetric with a real LLM call
latency_metric = LatencyMetric(max_seconds=10.0)

question = "What is the capital of France? Answer in one word."
answer, elapsed = ask(question)

tc_real = LLMTestCase(
    input=question,
    actual_output=answer,
)
tc_real.latency = elapsed  # attach latency in seconds

latency_metric.measure(tc_real)
status = "PASS" if latency_metric.is_successful() else "FAIL"

print(f"Response  : {answer}")
print(f"Latency   : {elapsed:.3f}s  (budget: {latency_metric.max_seconds}s)")
print(f"Result    : {status}")

In [ ]:
# Demonstrate a FAILING latency case by simulating a slow response
# (We don't actually wait 30s — we just set latency manually on the test case.)

tc_slow = LLMTestCase(
    input="What is 2 + 2?",
    actual_output="4",
)
tc_slow.latency = 31.7  # pretend this response took 31.7 seconds

latency_metric.measure(tc_slow)
status = "PASS" if latency_metric.is_successful() else "FAIL"

print(f"Simulated latency : {tc_slow.latency}s  (budget: {latency_metric.max_seconds}s)")
print(f"Result            : {status}")
print(f"Reason            : {latency_metric.reason}")

---
## 4. Combining LLM-Judge Metrics with Rule-Based Metrics

In production you almost always want **both** kinds of checks:

- **LLM-judge metrics** (GEval, AnswerRelevancyMetric) catch semantic failures: off-topic answers, poor tone, incomplete reasoning.
- **Rule-based metrics** (BaseMetric, LatencyMetric) enforce hard constraints that shouldn't require LLM judgment: required keywords, response time SLAs, format rules.

The pattern below is a reusable **quality gate** function. It runs every metric, collects failures, and prints a clear summary.

---

In [ ]:
# Define the combined quality gate

def run_quality_gate(test_case: LLMTestCase, metrics: list) -> dict:
    """
    Run a list of deepeval metrics against a single test case.
    Returns a summary dict with overall pass/fail and per-metric results.
    """
    results = []
    for metric in metrics:
        metric.measure(test_case)
        results.append({
            "metric":  metric.__name__,
            "score":   round(metric.score, 3),
            "passed":  metric.is_successful(),
            "reason":  metric.reason,
        })

    overall_pass = all(r["passed"] for r in results)
    return {"overall": overall_pass, "details": results}


def print_gate_report(label: str, report: dict) -> None:
    overall_str = "ALL PASS" if report["overall"] else "GATE FAILED"
    print(f"\n=== Quality Gate: {label} === [{overall_str}]")
    print(f"  {'Metric':<25} {'Score':>6}  {'Pass?':<6}  Reason")
    print(f"  {'-'*25} {'-'*6}  {'-'*6}  {'-'*40}")
    for r in report["details"]:
        status = "PASS" if r["passed"] else "FAIL"
        reason_short = (r["reason"] or "")[:60]
        print(f"  {r['metric']:<25} {r['score']:>6.3f}  {status:<6}  {reason_short}")


print("Quality gate helpers defined.")

In [ ]:
# Define the metrics for our production quality gate
security_keywords  = ["encryption", "authentication", "firewall"]

gate_metrics = [
    AnswerRelevancyMetric(threshold=0.7),
    KeywordCoverageMetric(required_keywords=security_keywords, threshold=0.8),
    LatencyMetric(max_seconds=10.0),
    GEval(
        name="Conciseness",
        criteria=(
            "The response should be concise and not ramble or repeat itself. "
            "A high-scoring response gets to the point quickly. "
            "A low-scoring response is verbose, repetitive, or padded with filler."
        ),
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.6,
    ),
]

print(f"Gate has {len(gate_metrics)} metrics:")
for m in gate_metrics:
    print(f"  - {m.__name__}")

In [ ]:
# Case 1 — a response that should PASS all gates
good_output = (
    "Our system enforces encryption for all data at rest and in transit. "
    "Authentication is handled via OAuth 2.0 with MFA required for admin accounts. "
    "A dedicated firewall filters inbound traffic at the network perimeter."
)

tc_good = LLMTestCase(
    input="What security measures does your platform use?",
    actual_output=good_output,
)
tc_good.latency = 1.8   # fast response

report_good = run_quality_gate(tc_good, gate_metrics)
print_gate_report("GOOD RESPONSE", report_good)

In [ ]:
# Case 2 — a response that FAILS the keyword gate (and possibly others)
bad_output = (
    "We take security seriously and have implemented various industry best practices. "
    "Our team is always working hard to keep your data safe. "
    "You can trust that we care about your privacy."
)

tc_bad = LLMTestCase(
    input="What security measures does your platform use?",
    actual_output=bad_output,
)
tc_bad.latency = 2.1   # latency is fine, but content is vague

report_bad = run_quality_gate(tc_bad, gate_metrics)
print_gate_report("VAGUE RESPONSE", report_bad)

---
## Try It Yourself

**Write a GEval metric for Answer Completeness.**

A multi-part question (e.g., *"What is X, why does it matter, and how is it measured?"*) requires the model to address **all three parts**. A response that only answers the first part should score low.

Tasks:
1. Write a `GEval` with `name="Answer Completeness"` and a criteria string that captures this.
2. Create two `LLMTestCase` objects — one that answers all parts, one that answers only one.
3. Measure both and print the scores.
4. Does the metric distinguish them reliably?

**Starter code below — fill in the blanks:**

In [ ]:
# TODO: Write your Answer Completeness GEval

completeness_metric = GEval(
    name="Answer Completeness",
    criteria=(
        # TODO: Write your criteria here.
        # Hint: tell the judge to check whether EVERY sub-question is addressed.
        # Describe what a complete answer looks like vs. a partial one.
        "TODO: replace this string with your criteria."
    ),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7,
)

multi_part_question = (
    "What is overfitting in machine learning, "
    "why is it a problem, "
    "and how can it be prevented?"
)

# TODO: write a complete answer (addresses all three parts)
complete_answer = "TODO: replace with a complete answer."

# TODO: write a partial answer (addresses only one part)
partial_answer = "TODO: replace with a partial answer."

for label, output in [("complete", complete_answer), ("partial", partial_answer)]:
    tc = LLMTestCase(input=multi_part_question, actual_output=output)
    completeness_metric.measure(tc)
    status = "PASS" if completeness_metric.is_successful() else "FAIL"
    print(f"[{label.upper()}] score={completeness_metric.score:.2f} | {status}")
    print(f"  Reason: {completeness_metric.reason}")
    print()

---
## Summary — When to Use Each Metric Type

| Metric Type | Class | Use When | LLM Call? | Cost |
|---|---|---|---|---|
| **GEval** | `GEval` | You need semantic/qualitative judgment (tone, completeness, relevance, style) and can express it as a rubric in plain English | Yes | Medium |
| **Custom rule-based** | `BaseMetric` | Deterministic checks: keyword presence, JSON format, response length, regex patterns | No | Zero |
| **Latency gate** | `LatencyMetric` | Enforcing an SLA — responses must arrive within a time budget | No | Zero |
| **Built-in DeepEval** | `AnswerRelevancyMetric`, `FaithfulnessMetric`, etc. | Standard LLM quality dimensions where someone else already wrote the rubric | Yes | Medium |

### Decision flowchart

```
Can you write the check as a simple Python if-statement?
  YES → BaseMetric
  NO  → Does a standard deepeval metric already cover this?
           YES → Use the built-in metric
           NO  → Write a GEval with a clear criteria string
```

**Next:** Day 6 — Running deepeval in CI: pytest integration, `deepeval test run`, and GitHub Actions.

---